# Understanding Learning Curves

Now that you've learned how to implement fine-tuning using both the `Trainer` API and custom training loops, it's crucial to understand how to interpret the results. Learning curves are invaluable tools that help you evaluate your model's performance during training and identify potential issues before they reduce performance.


Let's explore how to read and interpret accuracy and loss curves, understand what different curve shapes tell us about our model's behavior, and learn how to address common traning issues.

## What are Learning Curves?

Learning curves are visual representations of your model's performance metrics over time during training. The two most important curves to monitor are:
- **Loss curves**: Show how the model's error (loss) changes over training steps or epochs
- **Accuracy curves**: Shows the percentage of correct predictions over training steps or epochs.

These curves help us understand whether our model is learning effectively and can guide us in making adjustments to improve performance. In Transformers, these metrics are individually computed for each batch and then logged to the disk. We can then use libraries like *Weights & Biases** to visualize these curves and track our model's performance over time.

### Loss Curves

The loss curve shows how the model's error decreases over time.

In a typical successful training run, you'll see a curve similar to the one below:

![](./resources/loss_curve.png)
*Image source: [Hugging Face LLM Course, Chapter 3.5](https://huggingface.co/learn/llm-course/chapter3/5?fw=pt)*

- **High initial loss**: The model starts without optimization, so predictions are initially poor.
- **Decreasing loss:** As training progresses, the loss should generally decrease
- **Convergence:** Eventually, the loss stabilizes at a low value, indicating that the model has learned the patterns in the data.

We can use the `Trainer` API to track these metrics and visualize them in a dashboard. Below is an example of how to do this with **Weights & Biases**.

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

import evaluate

metric=evaluate.load('glue','mrpc')

import numpy as np
def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Example of tracking loss during training with the Trainer
from transformers import Trainer, TrainingArguments
import wandb
# Initialize Weights & Biases for experiment tracking
wandb.init(project="transformer-fine-tuning", name="bert-mrpc-analysis")

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    logging_steps=10,  # Log metrics every 10 steps
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    report_to="wandb",  # Send logs to Weights & Biases
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train and automatically log metrics
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1
50,0.582700,0.598143,0.671569,0.726531
100,0.515500,0.481190,0.767157,0.851794
150,0.492800,0.444436,0.796569,0.868045
200,0.381500,0.396500,0.843137,0.891892
250,0.259600,0.362040,0.840686,0.881603
300,0.231800,0.407079,0.865196,0.904679
350,0.234100,0.340227,0.865196,0.903339
400,0.133400,0.422422,0.850490,0.897133
450,0.269100,0.340626,0.870098,0.905526
500,0.044000,0.524982,0.857843,0.902685


/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argum

TrainOutput(global_step=690, training_loss=0.28975238411322884, metrics={'train_runtime': 181.9147, 'train_samples_per_second': 60.49, 'train_steps_per_second': 3.793, 'total_flos': 428577075854640.0, 'train_loss': 0.28975238411322884, 'epoch': 3.0})

### Accuracy Curves

The accuracy curve shows the percentage of correct predictions over time. Unlike loss curves, accuracy curves should generally increase as the model learns and can typically include more steps than loss curve.

![](./resources/accuracy_curve.png)

*Image source: [Hugging Face LLM Course, Chapter 3.5](https://huggingface.co/learn/llm-course/chapter3/5?fw=pt)*

- **Start Low**: Initial accuracy should be low, as the model has not yet learned the patterns in the data
- **Increase with training**: Accuracy should generally improve as the model learns if it is able to learn the patterns in the data.
- **May show plateaus**: Accuracy often increases in discrete jumps rather than smoothly, as the model makes predictions that are close to the true labels

### Why Accuracy Curves are 'steppy'

Unlike loss, which is continuous, accuracy is calculated by comparing discrete predictions to true labels. Small improvements in model confidence might not change the final prediciton, causing accuracy to remain falt until a threshold is crossed.

## Convergence

Convergence occurs when the model's performance stabilizes and the loss and accuracy curves level off. This is a sign that the model has learned the patterns in the data and is ready to be used. In simple terms, we are aiming for the model to converge to a stable performance every time we train it.

![](./resources/convergence.png)

*Image source: [Hugging Face LLM Course, Chapter 3.5](https://huggingface.co/learn/llm-course/chapter3/5?fw=pt)*

Once models have converged, we can use them to make predictions on new data and refer to evaluation metrics to understand how well the model is performing.

## Interpreting Learning Curve Patterns

Different curve shapes reveal different aspects of your model's training. Let's examine the most common patterns and what they mean.

### Healthy Learning Curves

![](./resources/healthy_learning_curve.png)

*Image source: [Hugging Face LLM Course, Chapter 3.5](https://huggingface.co/learn/llm-course/chapter3/5?fw=pt)*

The illustration above displayes both the loss curve (on the left) and the corresponding accuracy cruve (on the right). These curves have distinct characteristics.

**The loss curve** shows the value of the model's *loss* over time. Initially, the loss is high and then it gradually decreases, indicating that the model is improving. A decrease in the loss value suggests that the model is making better predictions, as the loss represents the error between the predicted output and the true output.

**The accuracy curve** represents the model's accuracy over time. The accuracy curve beings at a low value and increases as training progresses. Accuracy measures the proporition of correctly classified instances. So as the accuracy curve rises, it signifies that the model is making more correct predctions.

One notable difference between the curves is the smoothness and the presence of "plateaus" on the accuracy curve. While the loss decreases smoothly, the plateaus on the accuracy curve indicate discrete jumps in accuracy instead of a continuous increase. This behavior is attributed to how accuracy is measured. The loss can improve if the model's output gets closer to the target, even if the final predicition is still incorrect. Accuracy, however, only improves when the prediction crosses the threshold to be correct.

For example, in a binary classifier distinguishing cats (0) from dogs (1), if the model predicts 0.3 for an image of a dog (true value 1), this is rounded to 0 and is an incorrect classification. If the next step it predicts 0.4, it's still incorrect. The loss will have decreased because 0.4 is closer to 1 than 0.3, but the accuracy remains unchanged, creating a plateau. The accuracy will only jump up when the model predicts a value greater than 0.5 that gets rounded to 1.

#### Characteristics of healthy curves:

- **Smoth decline in loss:** Both training and validation loss decrease steadily.
- **Close training/validation performance**: Small gap between training and validation metrics.
- **Convergence:** Curves level off, indicating the model has learned the patterns.

## Practical Examples

First, hf will highlight some approaches to monitor the learning curves during training. Below, they will break down the different patterns that can be observed in the learning curves.

### During Training

During the training process (after you've hit `trainer.train()`), you can monitor these key indicators:
1. **Loss convergence**: Is the loss still decreasing or has it plateaued?
2. **Overfitting signs**: Is validation loss starting to increase while training loss decreases?
3. **Learning rate:** Are the curves too erratic (LR too high) or too flat (LR too low)?
4. **Stability:** Are these sudden spikes or drops that indicate problems?

### Afer Training

After the training process is complete, you can analyze the complete curves to understand the model's performance.

1. **Final performance**: Did the model reach acceptable performance levels?
2. **Efficiency**: Could the same performance be achieved with fewer epochs?
3. **Generalization:** How close are training and validation performances?
4. **Trends:** Would additional training likely improve performance?

### Overfitting

Overfitting occurs when the model learns too much from the training data and is unable to generalize to different data (represented by the validation set).

![](./resources/overfitting.png)

*Image source: [Hugging Face LLM Course, Chapter 3.5](https://huggingface.co/learn/llm-course/chapter3/5?fw=pt)*

#### Symptoms:

- Training loss continues to decrease while validation loss increases or plateaus
- Large gap between training and validation accuracy
- Training accuracy much higher than validation accuracy

#### Solution for overfitting:

- **Regularization**: Add dropout, weight decay, or other regularization techniques.
- **Early stopping:** Stop training when validation performance stops improving
- **Data augmentation:** Increase training data diversity
- **Reduce model complexity:** Use a smaller model or fewer parameters

In the code sample below, they use early stopping to prevent overfitting. They set the `early_stopping_patience` to 3, which means that iff the validation loss does not improve for 3 consecutive epochs, the training will be stopped.

In [4]:
# Example of detecting overfitting with early stopping
from transformers import EarlyStoppingCallback

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Initialize Weights & Biases for experiment tracking
wandb.init(project="transformer-fine-tuning", name="bert-mrpc-analysis-early-stop-patience")

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    num_train_epochs=10,  # Set high, but we'll stop early
)

# Add early stopping to prevent overfitting
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# Train and automatically log metrics
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


eval/accuracy,▁▄▅▇▇██▇██▇▇▇
eval/f1,▁▆▇▇▇████████
eval/loss,█▅▄▃▂▃▁▃▁▆▇█▇
eval/runtime,▇▄▄▁▅▂▄▄▅▁▃▂█
eval/samples_per_second,▂▅▅█▄▇▄▅▄█▆▇▁
eval/steps_per_second,▂▅▅█▄▇▄▅▄█▆▇▁
train/epoch,▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇██
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇█████
train/grad_norm,▁▂▂▂▃▂▂▂▂▂▂▂▃▅▂▂▂▂▄▄▃▂▂▂▆▅▃▃▁▂▂▁▂▃▂▃▁▁▄█
train/learning_rate,████▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁
train/loss,██▇▇▇▇▆▇▆▆▅▅▅▅▃▄▃▃▃▃▄▃▃▃▃▄▃▂▂▁▂▁▁▁▂▂▂▂▂▁


/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1
100,No log,0.611775,0.757353,0.832487
200,No log,0.494050,0.772059,0.851675
300,No log,0.431930,0.823529,0.881579
400,No log,0.441682,0.850490,0.897479
500,0.513200,0.427710,0.843137,0.888889
600,0.513200,0.550631,0.838235,0.892157
700,0.513200,0.401415,0.840686,0.884956
800,0.513200,0.531880,0.850490,0.897133
900,0.513200,0.462555,0.835784,0.880143
1000,0.278700,0.796776,0.828431,0.882550


/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argum

TrainOutput(global_step=1000, training_loss=0.39592153930664065, metrics={'train_runtime': 163.3883, 'train_samples_per_second': 224.496, 'train_steps_per_second': 28.093, 'total_flos': 294287659865040.0, 'train_loss': 0.39592153930664065, 'epoch': 2.178649237472767})

### Underfitting

Underfitting occurs when the model is too simple to capture the underlying patterns in the data.
This can happen for several reasons:
- The model is too small or lacks capacity to learn the patterns
- The learning rate is too low, causing slow learning
- The dataset is too small or not representative of the problem
- The model is not properly regularized

![](./resources/underfitting.png)

*Image source: [Hugging Face LLM Course, Chapter 3.5](https://huggingface.co/learn/llm-course/chapter3/5?fw=pt)*

#### Symptoms:

- Both training and validation loss remain high
- Model performance plateaus early in training
- Training accuracy is lower than expected

#### Solutions for underfitting:

- **Increase model capacity:** Use a larger model or more parameters
- **Train longer:** Increase the number of epochs
- **Adjust learning rate:** Try different learning rates
- **Check data quality:** Ensure your data is properly proprocessed.

In the code sample below, they train for more epochs to see if the model can learn the patterns in the data.

In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    # -num_train_epochs=5,
    num_train_epochs=10,
)

### Erratic Learning Curves

Erratic learning curves occur when the model is not learning effectively. This can happen for several reasons:

- The learning rate is too high, causing the model to overshoot the optimal parameters
- The batch size is too small, causing the model to learn slowly
- The model is not properly regularized, causing it to overfit to the training data
- The dataset is not properly preprocessed, causing the model to learn from noise

![](./resources/erratic_learning_curves.png)

*Image source: [Hugging Face LLM Course, Chapter 3.5](https://huggingface.co/learn/llm-course/chapter3/5?fw=pt)*

#### Symptoms:

- Frequent fluctuations in loss or accuracy
- Curves show high variance or instability
- Performance oscillates without clear trend

Both training and validation curves show erratic behavior.

![](./resources/erratic_learning_curves_2.png)

*Image source: [Hugging Face LLM Course, Chapter 3.5](https://huggingface.co/learn/llm-course/chapter3/5?fw=pt)*

#### Solutions for erratic curves:

- **Lower learning rate:** Reduce step size for more stable training
- **Increase batch size:** Larger batches provide more stable gradients
- **Gradient clipping:** Prevent exploding gradients
- **Better data preprocessing**: Ensure consistent data quality.

## Key Takeaways

- Learning curves are essential tools for understanding model training progress
- Monitor both loss and accuracy curves, but remember they have different chracteristics
- Overfitting shows as diverging training/validation performance
- Underfitting shows as poor performance on both training and validation data
- Tools like Weights & Biases make it easy to track and analyze learning curves
- Early stopping and porper regularization can address most common training issues